# SentinelIQ — 04 BERT Log Classifier
Fine-tune BERT for binary log anomaly classification. Enable GPU accelerator.

In [ ]:
!git clone https://github.com/hasan-rajab/SentinelIQ.git 2>/dev/null || echo "Already cloned"
%cd /kaggle/working/SentinelIQ
import sys
sys.path.insert(0, '/kaggle/working/SentinelIQ')
!pip install transformers pyyaml torch scikit-learn -q


In [ ]:
!python data/simulated/pipeline.py --duration 180 --anomaly-rate 0.08


In [ ]:
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay

from ml.models.bert_log import SentinelBertLog

sns.set_theme(style="darkgrid")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

with open('configs/model_config.yaml') as f:
    cfg = yaml.safe_load(f)

def load_jsonl(path):
    return pd.DataFrame([json.loads(l) for l in open(path)])


## Load & Inspect Log Data

In [ ]:
logs_df = load_jsonl('data/simulated/logs.jsonl')
print(f"Total records : {len(logs_df)}")
print(f"Anomalies     : {logs_df['is_anomaly'].sum()} ({logs_df['is_anomaly'].mean():.1%})")
print()
print("Sample messages:")
for _, row in logs_df.sample(5).iterrows():
    label = "ANOMALY" if row['is_anomaly'] else "normal "
    print(f"  [{label}] {row['message'][:90]}")


In [ ]:
# Message length distribution
logs_df['msg_len'] = logs_df['message'].str.len()
plt.figure(figsize=(10, 4))
normal_len = logs_df[logs_df['is_anomaly'] == False]['msg_len']
anomaly_len = logs_df[logs_df['is_anomaly'] == True]['msg_len']
plt.hist(normal_len, bins=30, alpha=0.6, color='#2ecc71', label='Normal')
plt.hist(anomaly_len, bins=30, alpha=0.6, color='#e74c3c', label='Anomaly')
plt.title('Log Message Length Distribution')
plt.xlabel('Character length')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()


## Split Data

In [ ]:
y = logs_df['is_anomaly'].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    logs_df, y, test_size=cfg['training']['test_size'],
    random_state=42, stratify=y)

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train, y_train, test_size=cfg['training']['val_size'],
    random_state=42, stratify=y_train)

print(f"Train : {len(X_train_final)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Train anomaly rate: {y_train_final.mean():.2%}")


## Fine-tune BERT
> Make sure GPU is enabled: Settings → Accelerator → GPU T4 x2

In [ ]:
bert = SentinelBertLog(config=cfg['bert_log'])
bert.fit(X_train_final, val_df=X_val)


## Evaluate

In [ ]:
results = bert.evaluate(X_test, y_test)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

scores = bert.score(X_test)
RocCurveDisplay.from_predictions(y_test, scores, ax=axes[0], name='BERT')
axes[0].set_title(f"ROC Curve (AUC={results['roc_auc']})")

PrecisionRecallDisplay.from_predictions(y_test, scores, ax=axes[1], name='BERT')
axes[1].set_title(f"Precision-Recall (AP={results['avg_precision']})")

ConfusionMatrixDisplay(confusion_matrix=np.array(results['confusion_matrix']),
    display_labels=['Normal','Anomaly']).plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title('Confusion Matrix')

plt.suptitle('BERT Log Classifier', fontsize=14)
plt.tight_layout()
plt.show()


## Error Analysis

In [ ]:
y_pred = bert.predict(X_test)
X_test_copy = X_test.copy()
X_test_copy['predicted'] = y_pred
X_test_copy['score'] = scores

# False negatives — missed anomalies
fn = X_test_copy[(X_test_copy['is_anomaly'] == True) & (X_test_copy['predicted'] == 0)]
print(f"False Negatives (missed anomalies): {len(fn)}")
for _, row in fn.head(3).iterrows():
    print(f"  score={row['score']:.3f} | {row['message'][:80]}")

print()

# False positives — wrong flags
fp = X_test_copy[(X_test_copy['is_anomaly'] == False) & (X_test_copy['predicted'] == 1)]
print(f"False Positives (wrong flags): {len(fp)}")
for _, row in fp.head(3).iterrows():
    print(f"  score={row['score']:.3f} | {row['message'][:80]}")


## Save Model

In [ ]:
import os
os.makedirs('ml/saved_models', exist_ok=True)
bert.save('ml/saved_models', name='bert_log')
print("BERT model saved.")
